## Understanding LLM APIs and inference

In [1]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

MODEL = os.getenv("MODEL_NAME", "gemini-2.5-flash")

print("Gemini connected!")
print("Model:", MODEL)

Gemini connected!
Model: gemini-3.5-flash-lite


In [4]:
response = client.models.generate_content(
    model=MODEL,
    contents="Say hello in one sentence."
)

print(response.text)

Hello and welcome!


In [5]:
response = client.interactions.create(
    model="gemini-3.5-flash",
    input="What is effective personal communication?"
)

print(response.output_text)

**Effective personal communication** is the process of exchanging information, ideas, thoughts, and feelings between individuals in a way that the message is received, understood, and acted upon as intended. 

It is not merely about talking; it is a two-way street that relies just as much on **listening and emotional connection** as it does on speaking. When personal communication is effective, it builds trust, resolves conflict, deepens relationships, and prevents misunderstandings.

Here is a breakdown of what constitutes effective personal communication:

---

### 1. The Core Components
Effective communication consists of three main elements:
*   **Verbal (Words):** The actual words you choose. Effective communicators use clear, simple, and direct language tailored to the listener.
*   **Vocal (Tone of Voice):** *How* you say it. Tone, pitch, pace, and volume convey emotion and intent. A gentle message spoken in a harsh tone will be received as hostile.
*   **Non-Verbal (Body Langua

### System Instructions

In [7]:
system_prompt = """
You are SpeakWise AI, a professional communication coach.

Your responsibilities:
1. Improve grammar.
2. Improve sentence clarity.
3. Suggest professional alternatives.
4. Preserve the user's original meaning.
5. Explain important corrections briefly.
"""

response = client.models.generate_content(
    model=MODEL,
    contents="I am good in communication and I can handle any situations.",
    config={
        "system_instruction": system_prompt,
        "temperature": 0.7
    }
)

print(response.text)

Hello! I would be happy to help you polish that sentence for a resume, interview, or professional bio. 

Here are a few ways to improve it:

### 1. Polished & Natural (Best for general professional use)
> "I am **good at** communication and **can handle any situation**."

### 2. High-Impact & Professional (Best for resumes or interviews)
> "I possess **strong communication skills** and **excel at navigating challenging situations**."

---

### Key Corrections & Explanations:
* **"Good in" vs. "Good at":** In English, we use the preposition **"at"** when talking about skills or abilities (e.g., *good at communication*, *good at problem-solving*). 
* **"Any situations" vs. "Any situation":** "Any" is typically followed by a singular noun when referring to possibilities in general (e.g., *any situation*, *any challenge*). 
* **Vocabulary Upgrade:** Replacing "handle" with words like **"navigate"** or **"manage"** sounds more refined and executive in a workplace context.


### Structured Output using Pydantic

In [8]:
from pydantic import BaseModel, Field
from typing import Literal

class CommunicationAnalysis(BaseModel):
    original_text: str
    corrected_text: str

    tone: Literal[
        "professional",
        'casual',
        "formal",
        "friendly"
    ]

    grammar_issues: list[str]

    clarity_score: int = Field(ge=0, le=10)

    suggestions: list[str]

In [10]:
response = client.models.generate_content(
    model = MODEL,
    contents="Analyze this sentencceL I am good in communication and I can handle any situations.",
    config={
        "response_mime_type": "application/json",
        "response_schema": CommunicationAnalysis,
        "temperature":0.2
    }
)

result = CommunicationAnalysis.model_validate_json(response.text)
print(result)

original_text='I am good in communication and I can handle any situations.' corrected_text='I am a strong communicator and can handle any situation.' tone='professional' grammar_issues=["Preposition usage: 'good in communication' should be 'good at communication' or 'a strong communicator'.", "Pluralization mismatch: 'any situations' should be singular 'any situation'."] clarity_score=7 suggestions=["Use 'good at' instead of 'good in'.", "Change 'any situations' to the singular 'any situation' for grammatical correctness.", "Consider rephrasing to 'I am a strong communicator and can handle any situation' for a more impactful tone."]


In [12]:
print(result.tone)
print(result.clarity_score)
print(result.suggestions)

professional
7
["Use 'good at' instead of 'good in'.", "Change 'any situations' to the singular 'any situation' for grammatical correctness.", "Consider rephrasing to 'I am a strong communicator and can handle any situation' for a more impactful tone."]


## Intent Detector

In [13]:
from pydantic import BaseModel, Field
from typing import Literal

class IntentClassification(BaseModel):

    intent: Literal[
        "email_writing",
        "interview_practice",
        "grammar_correction",
        "tone_improvement",
        "public_speaking",
        "conflict_resolution",
        "customer_communication"
    ]

    confidence: floar = Field(ge=0, le=1)

    reasoning: str